
# Thí nghiệm 02 — Dữ liệu áp suất bề mặt CMIP6

## 1. Khảo sát dữ liệu Surface Pressure

### 1.1. Mục tiêu

Notebook này thực hiện trích xuất chuỗi thời gian áp suất bề mặt từ dữ liệu khí hậu khu vực châu Âu, sử dụng các vị trí không gian đã được xác định trong thí nghiệm tốc độ gió CMIP6.

Quy trình gồm:

1. Đọc dữ liệu áp suất từ tệp NetCDF.
2. Lựa chọn 400 thời điểm đầu tiên.
3. Đối chiếu hệ tọa độ giữa dữ liệu gió và dữ liệu áp suất.
4. Xác định lại chính xác 5 vị trí đã sử dụng trong thí nghiệm Wind.
5. Trích xuất chuỗi áp suất tại các vị trí tương ứng.
6. Lưu dữ liệu dưới định dạng PyTorch để phục vụ huấn luyện.

### 1.2. Nguồn dữ liệu

Tệp dữ liệu:

`ps_EUR-11_MPI-M-MPI-ESM-LR_rcp26_r1i1p1_CLMcom-BTU-CCLM4-8-17_v1_3hr_202001010000-202012312100.nc`

Các thuộc tính chính:

| Thuộc tính | Giá trị |
|---|---|
| Dự án | CORDEX |
| Miền khí hậu | EUR-11 |
| Kịch bản | RCP2.6 |
| Năm dữ liệu | 2020 |
| Tần suất | 3 giờ |
| Biến sử dụng | `ps` |
| Đại lượng | Áp suất bề mặt (Surface Pressure) |
| Đơn vị | Pa |

Dữ liệu thuộc sản phẩm mô phỏng khí hậu khu vực, không phải số liệu quan trắc trực tiếp từ trạm khí tượng.

### 1.3. Cấu trúc dữ liệu

Biến `ps` sử dụng ba chiều:

- `time`: thời gian.
- `rlat`: tọa độ vĩ độ trong hệ lưới xoay.
- `rlon`: tọa độ kinh độ trong hệ lưới xoay.

Notebook lựa chọn 400 thời điểm đầu tiên, tạo tensor:

$$
P\in\mathbb{R}^{400\times412\times424}
$$

Trong đó $P_{t,i,j}$ biểu diễn áp suất bề mặt tại thời điểm $t$ và ô lưới $(i,j)$.

### 1.4. Liên hệ với dữ liệu CMIP6 Wind

Notebook sử dụng kết quả từ:

`01_climate_data.ipynb`

Cụ thể, tệp:

`data/wind_nearest_5sites.pt`

chứa chuỗi tốc độ gió đã trích xuất tại 5 vị trí không gian.

Mục tiêu là sử dụng đúng những ô lưới này để lấy áp suất bề mặt, giúp hai bộ dữ liệu có cùng hệ vị trí và thứ tự kênh.

### 1.5. Nguyên tắc bảo toàn dữ liệu

Dữ liệu đã được trích xuất và lưu từ lần thực nghiệm trước.

Việc tái cấu trúc notebook chỉ nhằm chuẩn hóa mã nguồn và giải thích phương pháp, không tạo lại dữ liệu hoặc thay đổi các vị trí đã chọn.


In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import torch


# 1. Xác định thư mục gốc của project
project_root = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "experiments" / "02_climate").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc của project."
    )


# 2. Xác định đường dẫn dữ liệu và thư mục đầu ra
experiment_dir = project_root / "experiments" / "02_climate"
data_dir = project_root / "datasets" / "climate" / "raw"
output_dir = experiment_dir / "data"

output_dir.mkdir(parents=True, exist_ok=True)


# 3. Tìm tệp NetCDF chứa dữ liệu Surface Pressure
pressure_path = next(
    data_dir.glob("ps_EUR-11_*.nc"),
    None
)

if pressure_path is None:
    raise FileNotFoundError(
        f"Không tìm thấy tệp Surface Pressure trong: {data_dir}"
    )


# 4. Mở dữ liệu NetCDF
ds = xr.open_dataset(pressure_path)


# 5. Lựa chọn 400 thời điểm đầu tiên
pressure = ds["ps"].isel(time=slice(0, 400))


# 6. Kiểm tra thông tin dữ liệu
print("=== CMIP6 SURFACE PRESSURE ===")

print("Dataset:", pressure_path.name)
print("Variable:", pressure.name)
print("Shape:", pressure.shape)
print("Dimensions:", pressure.dims)
print("Units:", pressure.attrs.get("units"))
print("Time steps:", pressure.sizes["time"])

Dataset: ps_EUR-11_MPI-M-MPI-ESM-LR_rcp26_r1i1p1_CLMcom-BTU-CCLM4-8-17_v1_3hr_202001010000-202012312100.nc
Variable: ps
Shape: (400, 412, 424)
Dimensions: ('time', 'rlat', 'rlon')
Units: Pa
Time steps: 400



## 2. Đồng bộ vị trí không gian giữa Wind và Surface Pressure

### 2.1. Mục tiêu

Để so sánh các thực nghiệm khí hậu, dữ liệu Surface Pressure được trích xuất tại cùng 5 vị trí không gian đã sử dụng trong thí nghiệm CMIP6 Wind.

Các chỉ số vị trí:

$$
I=\{0,5000,10000,15000,20000\}
$$

Đây là những chỉ số được lựa chọn trên lưới tọa độ mục tiêu trong `01_climate_data.ipynb`.

Notebook không lựa chọn ngẫu nhiên năm vị trí mới mà tìm lại các ô lưới thực tế tương ứng với dữ liệu gió đã lưu.

### 2.2. Kiểm tra tính tương thích của lưới

Trước khi trích xuất áp suất, notebook đối chiếu hai chiều tọa độ:

- `rlat`.
- `rlon`.

Hai mảng tọa độ phải giống nhau giữa dữ liệu Wind và Surface Pressure.

Điều kiện này được kiểm tra bằng `numpy.array_equal()`.

Việc hai tệp sử dụng cùng lưới cho phép áp dụng cặp chỉ số hàng và cột tìm được từ dữ liệu gió sang dữ liệu áp suất.

### 2.3. Xác định lại ô lưới bằng chuỗi tốc độ gió

Notebook sử dụng một phương pháp đối chiếu dựa trên giá trị dữ liệu đã lưu.

Các thời điểm được lựa chọn:

$$
T=\{0,1,2,3,4,10,20,30\}
$$

Với mỗi ô lưới, tạo một vector đặc trưng từ các giá trị tốc độ gió tại 8 thời điểm này.

Vector của một ô lưới có dạng:

$$
s_{i,j}\in\mathbb{R}^{8}
$$

Tương tự, với mỗi vị trí trong bộ dữ liệu Wind đã lưu, tạo vector đặc trưng từ 8 giá trị tốc độ gió tương ứng.

### 2.4. Tìm ô lưới bằng KD-tree

Các vector đặc trưng của toàn bộ ô lưới được đưa vào cấu trúc `cKDTree`.

Với mỗi vector của vị trí mục tiêu, thuật toán tìm vector gần nhất trong không gian 8 chiều.

Kết quả là chỉ số ô lưới ứng viên trong dữ liệu NetCDF.

Sau đó, notebook xác minh sự tương đồng giữa hai vector bằng `numpy.allclose()` với:

- `rtol = 1e-5`.
- `atol = 1e-5`.

Nếu các giá trị không khớp trong sai số cho phép, chương trình phát sinh lỗi để tránh trích xuất áp suất tại vị trí không chính xác.

**Lưu ý:** Đây là phương pháp nhận dạng ô lưới dựa trên chữ ký chuỗi thời gian, không phải phép tìm khoảng cách địa lý trực tiếp. Nó dựa vào dữ liệu gió đã lưu và kiểm tra kết quả tìm được.

### 2.5. Kết quả xác định vị trí

Các ô lưới đã được tìm thấy:

| Vị trí | Grid row | Grid column |
|---|---:|---:|
| 0 | 187 | 184 |
| 5000 | 246 | 202 |
| 10000 | 234 | 209 |
| 15000 | 222 | 216 |
| 20000 | 210 | 225 |

Các chỉ số này được sử dụng để trích xuất biến `ps` tại từng thời điểm.

### 2.6. Trích xuất chuỗi áp suất

Sau khi xác định các cặp chỉ số hàng và cột, notebook áp dụng phép chọn theo cặp:

`pressure.isel(rlat=site_rows, rlon=site_cols)`

Kết quả được chuyển về dạng:

$$
P_{\mathrm{raw}}\in\mathbb{R}^{400\times5}
$$

Trong đó:

- Mỗi hàng tương ứng một thời điểm.
- Mỗi cột tương ứng một trong năm vị trí.
- Giá trị được lưu bằng kiểu `float32`.
- Đơn vị áp suất vẫn là Pascal (Pa).

### 2.7. Kiểm tra dữ liệu

Trước khi lưu, notebook xác minh:

1. Ma trận kết quả có kích thước `(400, 5)`.
2. Tất cả giá trị đều hữu hạn.
3. Thứ tự năm vị trí được bảo toàn.

Dữ liệu được lưu dưới tên `pressure_raw`.

Ở bước này chưa thực hiện Min-Max scaling hoặc tạo cửa sổ dự báo. Các thao tác đó được triển khai riêng trong `05_climate_pressure_training.ipynb`.


In [ ]:
from scipy.spatial import cKDTree

# 1. Xác định tệp dữ liệu gió gốc và dataset đã lưu
wind_path = data_dir / "Klima_Daten_10m_3h_2020_RCP26.nc"
wind_pt_path = output_dir / "wind_nearest_5sites.pt"

wind_saved = torch.load(
    wind_pt_path,
    map_location="cpu",
    weights_only=True
)

wind_raw = np.asarray(wind_saved["wind_raw"])
location_indices = [0, 5000, 10000, 15000, 20000]


# 2. Xác định lại các ô lưới từ chuỗi tốc độ gió
sample_times = [0, 1, 2, 3, 4, 10, 20, 30]

with xr.open_dataset(wind_path) as wind_ds:

    # Kiểm tra hai bộ dữ liệu sử dụng cùng lưới tọa độ
    assert np.array_equal(
        ds["rlat"].values,
        wind_ds["rlat"].values
    )

    assert np.array_equal(
        ds["rlon"].values,
        wind_ds["rlon"].values
    )

    # Lấy dữ liệu gió tại các thời điểm dùng để đối chiếu
    wind_snapshots = wind_ds["sfcWind"].isel(
        time=sample_times
    ).values

    n_lat, n_lon = wind_snapshots.shape[1:]

    # Vector đặc trưng 8 chiều cho từng ô lưới
    source_signatures = wind_snapshots.reshape(
        len(sample_times), -1
    ).T

    # Vector đặc trưng cho 5 vị trí đã lưu
    target_signatures = wind_raw[sample_times, :].T

    # Tìm ô lưới có chuỗi tốc độ gió gần nhất
    tree = cKDTree(source_signatures)
    distances, flat_indices = tree.query(
        target_signatures
    )

    # Xác minh kết quả ánh xạ
    if not np.allclose(
        source_signatures[flat_indices],
        target_signatures,
        rtol=1e-5,
        atol=1e-5
    ):
        raise ValueError(
            "Cannot match saved wind locations "
            "to original NetCDF grid."
        )

    # Chuyển chỉ số phẳng về hàng và cột trên lưới
    rows, cols = np.unravel_index(
        flat_indices,
        (n_lat, n_lon)
    )


# 3. Trích xuất áp suất tại đúng 5 ô lưới
site_rows = xr.DataArray(rows, dims="site")
site_cols = xr.DataArray(cols, dims="site")

pressure_raw = (
    pressure.isel(
        rlat=site_rows,
        rlon=site_cols
    )
    .transpose("time", "site")
    .values
    .astype(np.float32)
)


# 4. Kiểm tra kích thước và tính hữu hạn của dữ liệu
print("Pressure shape:", pressure_raw.shape)

assert pressure_raw.shape == (400, 5)
assert np.isfinite(pressure_raw).all()


# 5. Lưu dataset để sử dụng trong notebook Training
save_path = output_dir / "pressure_nearest_5sites.pt"

torch.save({
    "pressure_raw": torch.from_numpy(pressure_raw),
    "location_indices": location_indices,
    "grid_rows": rows.tolist(),
    "grid_cols": cols.tolist(),
    "units": "Pa",
}, save_path)


# 6. Tổng hợp kết quả trích xuất
print("=== SURFACE PRESSURE DATASET ===")
print("Shape:", pressure_raw.shape)
print("Locations:", location_indices)
print("Grid rows:", rows.tolist())
print("Grid cols:", cols.tolist())

print(
    "Pressure range (Pa):",
    round(float(pressure_raw.min()), 2),
    "-",
    round(float(pressure_raw.max()), 2)
)

print("Saved:", save_path)

Pressure shape: (400, 5)
=== SURFACE PRESSURE DATASET ===
Shape: (400, 5)
Locations: [0, 5000, 10000, 15000, 20000]
Grid rows: [187, 246, 234, 222, 210]
Grid cols: [184, 202, 209, 216, 225]
Pressure range (Pa): 91840.41 - 103480.45
Saved: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\pressure_nearest_5sites.pt



## 3. Tổng kết dữ liệu Surface Pressure

### 3.1. Kết quả trích xuất

Notebook đã trích xuất thành công dữ liệu áp suất bề mặt tại 5 vị trí không gian tương ứng với thí nghiệm CMIP6 Wind.

Kết quả đã ghi nhận:

| Thuộc tính | Giá trị |
|---|---|
| Số thời điểm | 400 |
| Số vị trí | 5 |
| Kích thước dữ liệu | (400, 5) |
| Kiểu dữ liệu | float32 |
| Đơn vị | Pa |
| Áp suất nhỏ nhất | 91840.41 Pa |
| Áp suất lớn nhất | 103480.45 Pa |

Dữ liệu đã vượt qua các phép kiểm tra kích thước và tính hữu hạn trong notebook.

### 3.2. Nội dung tệp đã lưu

Tệp đầu ra:

`data/pressure_nearest_5sites.pt`

Bao gồm các trường:

| Trường | Nội dung |
|---|---|
| `pressure_raw` | Tensor áp suất bề mặt, kích thước (400, 5) |
| `location_indices` | Chỉ số của 5 vị trí đã chọn |
| `grid_rows` | Chỉ số hàng của các ô lưới |
| `grid_cols` | Chỉ số cột của các ô lưới |
| `units` | Đơn vị áp suất: Pa |

### 3.3. Vai trò trong quá trình huấn luyện

Dữ liệu được sử dụng trong:

`05_climate_pressure_training.ipynb`

Notebook tiếp theo thực hiện:

1. Đọc tensor `pressure_raw`.
2. Chuẩn hóa Min-Max.
3. Tạo các cửa sổ đầu vào và nhãn dự báo.
4. Khởi tạo các mô hình LSTM, PatchTST, Autoformer và Informer.
5. Huấn luyện và tính toán các chỉ số đánh giá.
6. Lưu checkpoint, kết quả và các hình minh họa.

Lưu ý rằng `pressure_raw` vẫn được biểu diễn bằng đơn vị Pascal.

Các giá trị đã chuẩn hóa chỉ được tạo trong notebook huấn luyện, không được lưu thay thế dữ liệu gốc tại đây.

### 3.4. Giới hạn của dữ liệu

Dữ liệu chỉ bao gồm:

- 400 thời điểm đầu tiên của tệp Surface Pressure.
- 5 vị trí không gian đã được xác định từ thí nghiệm Wind.
- Một biến khí hậu duy nhất là `ps`.

Do đó, kết quả huấn luyện từ bộ dữ liệu này chỉ phản ánh cấu hình và phạm vi dữ liệu đã lựa chọn.

Việc sử dụng cùng năm ô lưới giúp bảo đảm sự thống nhất vị trí không gian giữa Wind và Pressure, nhưng không đồng nghĩa với việc hai thí nghiệm sử dụng cùng đại lượng vật lý hoặc có thể so sánh trực tiếp giá trị sai số.

### 3.5. Bảo toàn dữ liệu thực nghiệm

Bộ dữ liệu `pressure_nearest_5sites.pt` đã được lưu từ lần trích xuất trước.

Trong quá trình tái cấu trúc repository:

- Không chạy lại các cell tạo dữ liệu.
- Không thay đổi thứ tự hoặc tọa độ của 5 vị trí.
- Không thay đổi số thời điểm được lựa chọn.
- Không chuẩn hóa hoặc biến đổi lại các giá trị đã lưu.
- Không ghi đè tệp dữ liệu hiện có.

Notebook được giữ lại để tài liệu hóa đầy đủ nguồn dữ liệu, phương pháp trích xuất và khả năng tái lập thực nghiệm.